# KA SS 26 · Challenge 2 — **Stage 10: seed-erosion probe**

Kaggle notebook. Run **interactively** on `GPU T4 x2`, `Persistence: Files only`, `Internet: On`.

### The one question this answers

Stage 8 trained two members and the ensemble was not where the gain came from:

| member | `SEED_EROSION_FRAC` | seed | crop | local F1 |
|---|---|---|---|---|
| n1 | 0.45 | 2026 | 448×576 | 0.7962 |
| n2 | **0.58** | 43 | 400×512 | **0.8055** |
| ens(n1,n2) | — | — | — | 0.8073 |

**+0.0093 from one member to the other** — larger than the SimSiam adaptation, larger than any
ensemble built so far (Stage 7's three members bought +0.0036; Stage 9's five-member cross-family
blend bought −0.0001). And it points where every diagnostic has pointed: predictions/image ran
107.8 → 111.3 against a GT mean of **125.2**. Still under-counting by ~11%.

Raising the erosion fraction shrinks each training seed core, which separates touching cells and
produces more instances. `t_seed` is already a confirmed *interior* optimum (0.65 won in Stage 5
with a grid topping at 0.65, and won again in Stage 7 with 0.90 above it), so the decode threshold
is not the binding constraint. The **target** is. Nobody has tested past 0.58.

### Why this run is designed the way it is

n1 → n2 changed **three** things at once: erosion, seed, and crop. That gain is confounded. Every
probe here fixes **seed 43 and crop 400×512** — n2's values — so the only variable is erosion and
the result is an actual curve.

| probe | erosion | why |
|---|---|---|
| `e70` | 0.70 | extends the trend furthest; run first because it is the most informative single point |
| `e65` | 0.65 | fills in between 0.58 and 0.70 |
| `e80` | 0.80 | brackets the far side — if this collapses, the optimum is bracketed |
| `e45` | 0.45 | control at n2's seed/crop, de-confounding the original n1→n2 comparison |

Budget-guarded and ordered by information value: if time runs out after two, the two that ran are
the two worth having.

### What it does not do

No new ensemble members for their own sake, no detector, no architecture change. Stage 9 measured
the ensemble return at −0.0001 and four independent Mask R-CNN runs capped ~0.03 below watershed.
The ensemble here is opportunistic only: once the maps are cached, greedy forward selection over
**all** available members (Stage 8's n1/n2 plus everything trained here) is nearly free.

### Reading the result

Public differences under ~0.008 are inside the noise of a 91-image split — Stage 5, 7, 8 and 9
landed at 0.5293 / 0.5259 / 0.5294 / 0.5286 across four genuinely different models. Trust the
**local curve shape** here, not a single public score. If F1 rises to 0.70 and falls at 0.80, that
is a real optimum; if it is flat, the knob is exhausted and no further probe will help.

## Before Run All

1. **Session options → Accelerator: GPU T4 x2, Internet: On, Persistence: Files only.**
2. **Add Input → Competitions → KA SS 26: Challenge 2.**
3. **Add Input → Datasets → `stage_8_nonssl`** (supplies n1/n2 maps for the ensemble step and
   `val_ids.json` for the split check).
4. Run interactively — **not** Save & Run All, which is what killed Stage 3.
5. `RESET_WORKDIR` stays **False**.

Cells 3 and 8 are the fail-fast gates and clear inside two minutes. Download the checkpoints from
`/kaggle/working/ckpt` as they appear.

In [1]:
# ── 1 · Imports and environment ───────────────────────────────────────────────
import os, sys, gc, json, math, time, random, shutil, subprocess, warnings, itertools
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import scipy.ndimage as ndi
from scipy.ndimage import distance_transform_edt, binary_dilation
from skimage.segmentation import watershed

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm

try:
    import cv2
    HAVE_CV2 = True
except ImportError:
    HAVE_CV2 = False

try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = False

def amp_autocast():
    return _autocast("cuda", dtype=torch.float16) if _AMP_NEW else _autocast()

def make_scaler():
    return _GradScaler("cuda") if _AMP_NEW else _GradScaler()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
assert torch.cuda.is_available(), "No GPU. Session options -> Accelerator -> GPU T4 x2."

T_START = time.time()
def elapsed():
    return time.time() - T_START

def fmt(sec):
    sec = int(sec)
    return f"{sec // 3600}h {(sec % 3600) // 60:02d}m {sec % 60:02d}s"

print("torch", torch.__version__, "| timm", timm.__version__, "| opencv", HAVE_CV2)
print("gpu:", torch.cuda.get_device_properties(0).name)

torch 2.10.0+cu128 | timm 1.0.26 | opencv True
gpu: Tesla T4


In [2]:
# ── 2 · Configuration ─────────────────────────────────────────────────────────
class CFG:
    RUN_TAG = "stage10_erosion"

    SMOKE_TEST = False      # ~10 min rehearsal: 1 probe, 3 epochs, tiny grids
    RESUME = True
    RESET_WORKDIR = False   # DANGER: wipes ckpt/ and maps/. Never True while recovering.
    STRICT_SPLIT = True

    # ── wall-clock budgets ────────────────────────────────────────────────────
    TOTAL_BUDGET_SEC  = 5.0 * 3600
    MEMBER_BUDGET_SEC = 45 * 60
    FINAL_RESERVE_SEC = 45 * 60

    IMG_H, IMG_W = 520, 704
    PAD_MULT = 32

    # ── split: must reproduce Stage 7/8 exactly or the ensemble is dishonest ──
    SPLIT_SEED = 2026
    GROUP_SPLIT = True
    DUP_CORR_THR = 0.72
    N_FOLDS = 5
    FOLD = 0
    EXPECT_TRAIN, EXPECT_VAL, EXPECT_GROUPS = 338, 86, 421

    ENCODER = "convnext_tiny.fb_in22k_ft_in1k"
    ENCODER_FALLBACKS = ("convnext_tiny", "resnet50", "resnet34")
    DECODER_CHANNELS = (256, 128, 64, 32)
    OUT_CHANNELS = 2

    BORDER_DILATION = 2
    BORDER_WEIGHT = 4.0

    # ── supervised schedule: Stage 5, unchanged ───────────────────────────────
    SUP_EPOCHS = 70
    SUP_BATCH = 8
    SUP_LR = 3e-4
    SUP_ENCODER_LR_MULT = 0.25
    SUP_WD = 1e-4
    FREEZE_ENCODER_EPOCHS = 2
    WARMUP_STEPS = 150
    STEPS_PER_EPOCH = 42
    EVAL_START = 24
    EVAL_EVERY = 2
    GRAD_CLIP = 5.0

    # SEED AND CROP ARE FIXED AT n2's VALUES ACROSS EVERY PROBE.
    # n1 -> n2 moved erosion, seed and crop together, so its +0.0093 is confounded.
    # Holding the other two still is what turns this into a curve instead of anecdotes.
    PROBE_SEED = 43
    PROBE_CROP = (400, 512)
    MEMBERS = [
        dict(name="e70", erosion=0.70),   # most informative single point - runs first
        dict(name="e65", erosion=0.65),
        dict(name="e80", erosion=0.80),
        dict(name="e45", erosion=0.45),   # control: de-confounds the original n1 vs n2
    ]

    AUG_HFLIP = 0.5
    AUG_VFLIP = 0.5
    AUG_SCALE = (0.85, 1.20)
    AUG_BRIGHT = 0.15
    AUG_CONTRAST = 0.15
    AUG_NOISE = 0.02

    TTA_FLIPS = True
    TTA_SCALES = (0.85, 1.0, 1.15)

    # score_thr was degenerate in Stage 5 and chose 0.0 unaided in Stages 7-9. Axis dropped.
    # min_area reaches down to 30 because higher erosion produces smaller seeds, and
    # GT area p1 is 53.
    T_CELL_GRID = [0.40, 0.45, 0.50, 0.55, 0.60]
    T_SEED_GRID = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
    MIN_AREA_GRID = [30, 50, 70, 90, 110]
    SCORE_THR_GRID = [0.0]
    COARSE_TC = [0.45, 0.50, 0.55]
    COARSE_TS = [0.55, 0.60, 0.65, 0.70]
    COARSE_MA = [50, 90]
    SWEEP_SUBSET = 86
    ROBUST_TOL = 0.004

    DEFAULT_POST = dict(t_cell=0.50, t_seed=0.65, min_area=70, score_thr=0.0)
    SUB_INDEX_COL = "idx"

    STAGE8_TAG = "stage8_nonssl"    # n1 / n2 maps come from the attached dataset


if CFG.SMOKE_TEST:
    CFG.SUP_EPOCHS = 3
    CFG.EVAL_START = 1
    CFG.EVAL_EVERY = 1
    CFG.MEMBERS = CFG.MEMBERS[:1]
    CFG.TTA_SCALES = (1.0,)
    CFG.T_CELL_GRID = [0.45, 0.50]
    CFG.T_SEED_GRID = [0.60, 0.65]
    CFG.MIN_AREA_GRID = [50, 90]
    CFG.COARSE_TC = [0.50]
    CFG.COARSE_TS = [0.60, 0.65]
    CFG.SWEEP_SUBSET = 12
    CFG.MEMBER_BUDGET_SEC = 10 * 60
    CFG.FINAL_RESERVE_SEC = 12 * 60
    print(">>> SMOKE TEST - results are meaningless, this only proves the pipeline runs")


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SPLIT_SEED)
STATS = {"run_tag": CFG.RUN_TAG, "smoke_test": CFG.SMOKE_TEST}
print("run tag:", CFG.RUN_TAG)
print("probes:", [(m["name"], m["erosion"]) for m in CFG.MEMBERS],
      f"| all at seed {CFG.PROBE_SEED}, crop {CFG.PROBE_CROP}")

run tag: stage10_erosion
probes: [('e70', 0.7), ('e65', 0.65), ('e80', 0.8), ('e45', 0.45)] | all at seed 43, crop (400, 512)


In [3]:
# ── 3 · Paths, inputs, encoder preflight ──────────────────────────────────────
WORK = Path("/kaggle/working")
CKPT, ART, CACHE, MAPS = WORK / "ckpt", WORK / "artifacts", WORK / "cache", WORK / "maps"

if CFG.RESET_WORKDIR:
    print("!! RESET_WORKDIR is True - clearing ckpt/, artifacts/, maps/")
    for p in (CKPT, ART, MAPS):
        if p.exists():
            shutil.rmtree(p)
for p in (CKPT, ART, CACHE, MAPS):
    p.mkdir(parents=True, exist_ok=True)


def find_dataset_root():
    base = Path("/kaggle/input")
    assert base.exists(), "No /kaggle/input. Add Input -> Competitions -> KA SS 26: Challenge 2."
    for csv_path in sorted(base.rglob("train.csv")):
        d = csv_path.parent
        if (d / "train").is_dir() and (d / "test").is_dir():
            return d
    raise FileNotFoundError("Competition data not found. Add Input -> Competitions.")

BASE = find_dataset_root()
TRAIN_DIR, TEST_DIR, TRAIN_CSV = BASE / "train", BASE / "test", BASE / "train.csv"
print("data root:", BASE)
print("train", len(list(TRAIN_DIR.glob('*.png'))), "| test", len(list(TEST_DIR.glob('*.png'))))

# Stage 8 maps are optional - without them this still runs, just without n1/n2 in the pool.
S8_DIR = None
for cand in sorted(Path("/kaggle/input").glob("*")):
    hit = next(iter(cand.rglob(f"{CFG.STAGE8_TAG}_*_val.npz")), None)
    if hit is not None:
        S8_DIR = hit.parent
        break
if S8_DIR is None:
    print("!! stage_8_nonssl not attached - n1/n2 will not be in the ensemble pool.")
    STAGE8_MEMBERS = []
else:
    STAGE8_MEMBERS = sorted({p.name.split("_")[2]
                             for p in S8_DIR.glob(f"{CFG.STAGE8_TAG}_*_val.npz")})
    print("stage 8 dir:", S8_DIR, "| members:", STAGE8_MEMBERS)


def resolve_encoder():
    for name in [CFG.ENCODER] + list(CFG.ENCODER_FALLBACKS):
        try:
            m = timm.create_model(name, pretrained=True, features_only=True, in_chans=3)
            ch = m.feature_info.channels()
            del m; gc.collect()
            print(f"encoder ok: {name} | feature channels {ch}")
            return name
        except Exception as exc:
            print(f"  [{name}] unavailable: {type(exc).__name__}: {str(exc)[:160]}")
    raise RuntimeError("No pretrained encoder. Turn Internet ON in Session options.")

CFG.ENCODER = resolve_encoder()
STATS["encoder"] = CFG.ENCODER

data root: /kaggle/input/competitions/ka-ss-26-challenge-2/cells_segmentation
train 424 | test 182
stage 8 dir: /kaggle/input/datasets/ghostylicious/stage-8-nonssl | members: ['n1', 'n2']


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

encoder ok: convnext_tiny.fb_in22k_ft_in1k | feature channels [96, 192, 384, 768]


In [4]:
# ── 4 · Labels and RLE codec (byte-identical to Stages 5/7/8) ─────────────────
df_raw = pd.read_csv(TRAIN_CSV)
ID_COL = next(c for c in df_raw.columns if c.lower() in ("id", "image_id", "imageid"))
ANN_COL = next(c for c in df_raw.columns
               if c.lower() in ("annotation", "encodedpixels", "rle", "segmentation"))

TRAIN_IDS = sorted(p.stem for p in TRAIN_DIR.glob("*.png"))
TEST_IDS = sorted(p.stem for p in TEST_DIR.glob("*.png"))
with Image.open(TRAIN_DIR / f"{TRAIN_IDS[0]}.png") as im:
    CFG.IMG_W, CFG.IMG_H = im.size
PAD_H = int(math.ceil(CFG.IMG_H / CFG.PAD_MULT) * CFG.PAD_MULT)
PAD_W = int(math.ceil(CFG.IMG_W / CFG.PAD_MULT) * CFG.PAD_MULT)
NPIX = CFG.IMG_H * CFG.IMG_W


def rle_decode_flat(rle, h, w, order="C"):
    if not isinstance(rle, str) or not rle.strip() or rle.strip() == "-1":
        return np.empty(0, dtype=np.int64)
    values = np.asarray(rle.split(), dtype=np.int64)
    starts, lengths = values[0::2] - 1, values[1::2]
    idx = np.concatenate([np.arange(s, s + l) for s, l in zip(starts, lengths)])
    return idx if order == "C" else (idx % h) * w + (idx // h)


def rle_encode(mask_flat_u8):
    pixels = np.concatenate([[0], mask_flat_u8, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(int(x)) for x in runs)


def flat_to_rle(flat):
    mask = np.zeros(NPIX, dtype=np.uint8)
    mask[flat] = 1
    return rle_encode(mask)


probe_df = df_raw[df_raw[ANN_COL].astype(str) != "-1"].sample(min(200, len(df_raw)), random_state=0)
order_report = {}
for order in ("C", "F"):
    single = []
    for rle in probe_df[ANN_COL].astype(str):
        flat = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, order)
        if flat.size == 0 or flat.max() >= NPIX:
            continue
        m = np.zeros(NPIX, bool); m[flat] = True
        single.append(ndi.label(m.reshape(CFG.IMG_H, CFG.IMG_W))[1] == 1)
    order_report[order] = float(np.mean(single) * 100) if single else 0.0
RLE_ORDER = max(order_report, key=lambda o: order_report[o])
print("RLE_ORDER =", RLE_ORDER, {k: round(v, 1) for k, v in order_report.items()})
assert order_report[RLE_ORDER] > 90

INST = defaultdict(list)
for image_id, rle in zip(df_raw[ID_COL].astype(str), df_raw[ANN_COL].astype(str)):
    flat = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, RLE_ORDER)
    if flat.size:
        INST[image_id].append(np.unique(flat))
counts = np.array([len(INST[i]) for i in TRAIN_IDS])
areas = np.array([len(f) for i in TRAIN_IDS for f in INST[i]])
STATS["gt_mean_instances"] = float(counts.mean())
print(f"instances {int(counts.sum())} | per image mean {counts.mean():.1f} "
      f"p50 {int(np.percentile(counts, 50))} max {int(counts.max())}")
print(f"instance area p1 {int(np.percentile(areas, 1))} p50 {int(np.percentile(areas, 50))}")

bad = 0
for image_id in TRAIN_IDS[:40]:
    for flat in INST[image_id]:
        if not np.array_equal(
                np.unique(rle_decode_flat(flat_to_rle(flat), CFG.IMG_H, CFG.IMG_W, RLE_ORDER)), flat):
            bad += 1
assert bad == 0, f"RLE round-trip lossy on {bad} instances."
print(">>> RLE round-trip exact on 40 images.")

RLE_ORDER = C {'C': 91.5, 'F': 0.0}
instances 53065 | per image mean 125.2 p50 47 max 782
instance area p1 53 p50 215
>>> RLE round-trip exact on 40 images.


In [5]:
# ── 5 · Images and per-erosion target sets ────────────────────────────────────
def load_gray_u8(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L"), dtype=np.uint8)

IMAGES = {i: load_gray_u8(TRAIN_DIR / f"{i}.png") for i in TRAIN_IDS}
MEAN = float(np.mean([IMAGES[i].mean() for i in TRAIN_IDS]) / 255.0)
STD = float(np.mean([IMAGES[i].std() for i in TRAIN_IDS]) / 255.0)
print(f"loaded {len(IMAGES)} images | mean/std {MEAN:.4f}/{STD:.4f}")


def build_targets_for(image_id, frac):
    h, w = CFG.IMG_H, CFG.IMG_W
    fg = np.zeros((h, w), np.uint8)
    seed = np.zeros((h, w), np.uint8)
    overlap = np.zeros((h, w), np.uint16)
    for flat in INST[image_id]:
        ys, xs = np.divmod(flat, w)
        y0, y1 = ys.min(), ys.max() + 1
        x0, x1 = xs.min(), xs.max() + 1
        sub = np.zeros((y1 - y0, x1 - x0), bool)
        sub[ys - y0, xs - x0] = True
        fg[y0:y1, x0:x1] |= sub
        dist = distance_transform_edt(np.pad(sub, 1))[1:-1, 1:-1]
        dmax = dist.max()
        if dmax > 0:
            core = dist > frac * dmax
            if not core.any():          # a high frac can erase a small cell's core entirely -
                core = dist >= dmax     # this keeps at least the peak pixel, so no cell is lost
            seed[y0:y1, x0:x1] |= core
        pad = CFG.BORDER_DILATION
        ys0, ys1 = max(0, y0 - pad), min(h, y1 + pad)
        xs0, xs1 = max(0, x0 - pad), min(w, x1 + pad)
        big = np.zeros((ys1 - ys0, xs1 - xs0), bool)
        big[y0 - ys0:y1 - ys0, x0 - xs0:x1 - xs0] = sub
        overlap[ys0:ys1, xs0:xs1] += binary_dilation(big, iterations=pad).astype(np.uint16)
    return np.stack([fg, seed, (overlap >= 2).astype(np.uint8)], 0)


TARGETS = None
_TARGET_KEY = None
SEED_PCT = {}


def load_targets(frac):
    global TARGETS, _TARGET_KEY
    if _TARGET_KEY == frac:
        return
    TARGETS = None
    gc.collect()
    path = CACHE / f"targets_e{int(round(frac * 100)):03d}.npz"
    if CFG.RESUME and path.exists():
        with np.load(path) as z:
            TARGETS = {k: z[k] for k in z.files}
        print(f"targets (erosion {frac}) from cache")
    else:
        t0 = time.time()
        TARGETS = {i: build_targets_for(i, frac) for i in TRAIN_IDS}
        np.savez_compressed(path, **TARGETS)
        print(f"targets (erosion {frac}) built in {time.time() - t0:.0f}s")
    _TARGET_KEY = frac
    pct = float(np.mean([TARGETS[i][1].mean() for i in TRAIN_IDS]) * 100)
    SEED_PCT[frac] = pct
    # This is the knob doing its work: seed coverage must shrink as erosion rises.
    print(f"  seed channel covers {pct:.2f}% of pixels at erosion {frac}")


load_targets(CFG.MEMBERS[0]["erosion"])

loaded 424 images | mean/std 0.5018/0.0522
targets (erosion 0.7) built in 24s
  seed channel covers 1.56% of pixels at erosion 0.7


In [6]:
# ── 6 · Grouped split + identity check against Stage 8 ────────────────────────
TH_H, TH_W = 65, 88


def thumbnail_vec(arr):
    a = np.asarray(Image.fromarray(arr).resize((TH_W, TH_H), Image.BILINEAR), np.float32).ravel()
    a -= a.mean()
    n = np.linalg.norm(a)
    return a / n if n > 0 else a

THUMBS = np.stack([thumbnail_vec(IMAGES[i]) for i in TRAIN_IDS])
CORR = THUMBS @ THUMBS.T
np.fill_diagonal(CORR, 0.0)

adj = CORR >= CFG.DUP_CORR_THR
group_of = -np.ones(len(TRAIN_IDS), int)
g = 0
for i in range(len(TRAIN_IDS)):
    if group_of[i] >= 0:
        continue
    stack, group_of[i] = [i], g
    while stack:
        u = stack.pop()
        for v in np.nonzero(adj[u])[0]:
            if group_of[v] < 0:
                group_of[v] = g
                stack.append(v)
    g += 1
n_groups = g

rng = np.random.RandomState(CFG.SPLIT_SEED)
order = rng.permutation(n_groups)
fold_of_group = {int(gid): int(k % CFG.N_FOLDS) for k, gid in enumerate(order)}
val_mask = np.array([fold_of_group[int(x)] == CFG.FOLD for x in group_of])
train_ids = [i for i, m in zip(TRAIN_IDS, val_mask) if not m]
val_ids = [i for i, m in zip(TRAIN_IDS, val_mask) if m]

vi, ti = np.nonzero(val_mask)[0], np.nonzero(~val_mask)[0]
max_corr = float(CORR[np.ix_(vi, ti)].max())
print(f"groups {n_groups} | train/val {len(train_ids)}/{len(val_ids)} | max corr {max_corr:.3f}")
assert max_corr < CFG.DUP_CORR_THR, "Near-duplicate leaked across the split."

got = (len(train_ids), len(val_ids), n_groups)
want = (CFG.EXPECT_TRAIN, CFG.EXPECT_VAL, CFG.EXPECT_GROUPS)
if got != want and CFG.STRICT_SPLIT:
    raise RuntimeError(f"SPLIT MISMATCH: got {got}, Stages 7/8 had {want}. "
                       "Local F1 would not be comparable and n1/n2 could not be ensembled.")

# Identity, not a count: Stage 7 matched Stage 5's configured split and still differed.
if S8_DIR is not None:
    vid = next(iter(S8_DIR.rglob("val_ids.json")), None)
    if vid is not None:
        s8_val = list(json.loads(vid.read_text())["val_ids"])
        if s8_val != list(val_ids):
            only8, only10 = set(s8_val) - set(val_ids), set(val_ids) - set(s8_val)
            raise RuntimeError(
                f"Validation sets differ from Stage 8 ({len(only8)} / {len(only10)} images).\n"
                "n1/n2 cannot join the ensemble pool - set STAGE8_MEMBERS = [] to continue alone.")
        print(">>> validation set identical to Stage 8 - n1/n2 can join the pool.")

STATS.update(n_groups=int(n_groups), max_val_train_corr=max_corr,
             n_train=len(train_ids), n_val=len(val_ids))
(ART / "val_ids.json").write_text(json.dumps(
    {"val_ids": val_ids, "n_train": len(train_ids), "n_groups": int(n_groups)}, indent=1))

groups 421 | train/val 338/86 | max corr 0.665
>>> validation set identical to Stage 8 - n1/n2 can join the pool.


1602

In [7]:
# ── 7 · Dataset and augmentation (unchanged from Stage 5/7/8) ─────────────────
def resize_pair(img, tgt, scale):
    h, w = img.shape
    nh, nw = max(64, int(round(h * scale))), max(64, int(round(w * scale)))
    if HAVE_CV2:
        img_r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
        tgt_r = np.stack([cv2.resize(tgt[c], (nw, nh), interpolation=cv2.INTER_NEAREST)
                          for c in range(tgt.shape[0])])
    else:
        img_r = np.asarray(Image.fromarray((img * 255).astype(np.uint8)).resize(
            (nw, nh), Image.BILINEAR), np.float32) / 255.0
        tgt_r = np.stack([np.asarray(Image.fromarray(tgt[c]).resize((nw, nh), Image.NEAREST))
                          for c in range(tgt.shape[0])])
    return img_r.astype(np.float32), tgt_r.astype(np.uint8)


def pad_to(arr, height, width, mode="reflect"):
    ph, pw = max(0, height - arr.shape[-2]), max(0, width - arr.shape[-1])
    if ph == 0 and pw == 0:
        return arr
    pad_width = [(0, 0)] * (arr.ndim - 2) + [(0, ph), (0, pw)]
    kw = {} if mode == "reflect" else {"constant_values": 0}
    return np.pad(arr, pad_width, mode=mode, **kw)


class CellDataset(Dataset):
    def __init__(self, ids, crop, training=True):
        self.ids = list(ids)
        self.crop_h, self.crop_w = crop
        self.training = bool(training)

    def __len__(self):
        return len(self.ids)

    def augment(self, img, tgt):
        if random.random() < CFG.AUG_HFLIP:
            img, tgt = img[:, ::-1], tgt[:, :, ::-1]
        if random.random() < CFG.AUG_VFLIP:
            img, tgt = img[::-1], tgt[:, ::-1]
        img = np.ascontiguousarray(img)
        tgt = np.ascontiguousarray(tgt)
        s = random.uniform(*CFG.AUG_SCALE)
        if abs(s - 1.0) > 1e-3:
            img, tgt = resize_pair(img, tgt, s)
        img = img + random.uniform(-CFG.AUG_BRIGHT, CFG.AUG_BRIGHT) * STD
        f = 1.0 + random.uniform(-CFG.AUG_CONTRAST, CFG.AUG_CONTRAST)
        img = (img - MEAN) * f + MEAN
        img = img + np.random.randn(*img.shape).astype(np.float32) * CFG.AUG_NOISE * STD
        return np.clip(img, 0.0, 1.0).astype(np.float32), tgt

    def random_crop(self, img, tgt):
        img = pad_to(img, self.crop_h, self.crop_w)
        tgt = pad_to(tgt, self.crop_h, self.crop_w, mode="constant")
        h, w = img.shape
        y = random.randint(0, h - self.crop_h)
        x = random.randint(0, w - self.crop_w)
        return img[y:y + self.crop_h, x:x + self.crop_w], tgt[:, y:y + self.crop_h, x:x + self.crop_w]

    def __getitem__(self, index):
        image_id = self.ids[index]
        img = IMAGES[image_id].astype(np.float32) / 255.0
        tgt = TARGETS[image_id].copy()
        if self.training:
            img, tgt = self.augment(img, tgt)
            img, tgt = self.random_crop(img, tgt)
        else:
            img = pad_to(img, PAD_H, PAD_W)
            tgt = pad_to(tgt, PAD_H, PAD_W, mode="constant")
        weight = 1.0 + CFG.BORDER_WEIGHT * tgt[2].astype(np.float32)
        x = torch.from_numpy(np.ascontiguousarray(img)).unsqueeze(0).repeat(3, 1, 1)
        y = torch.from_numpy(np.ascontiguousarray(tgt[:2]).astype(np.float32))
        w = torch.from_numpy(np.ascontiguousarray(weight)).unsqueeze(0)
        return x, y, w, image_id


def normalize_batch(x):
    return (x - MEAN) / max(STD, 1e-6)


_ds = CellDataset(train_ids, CFG.PROBE_CROP, True)
_x, _y, _w, _ = _ds[0]
print("sample shapes:", tuple(_x.shape), tuple(_y.shape))
del _ds, _x, _y, _w

sample shapes: (3, 400, 512) (2, 400, 512)


In [8]:
# ── 8 · ConvNeXt-UNet ─────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


def up_to(x, ref):
    return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)


class UNet(nn.Module):
    def __init__(self, encoder_name, out_ch=2, dec=(256, 128, 64, 32), pretrained=True):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=pretrained,
                                         features_only=True, in_chans=3)
        chs = self.encoder.feature_info.channels()[-4:]
        self.stem = nn.Sequential(
            nn.Conv2d(3, dec[3], 3, padding=1, bias=False),
            nn.BatchNorm2d(dec[3]), nn.ReLU(inplace=True))
        self.d3 = ConvBlock(chs[3] + chs[2], dec[0])
        self.d2 = ConvBlock(dec[0] + chs[1], dec[1])
        self.d1 = ConvBlock(dec[1] + chs[0], dec[2])
        self.d0 = ConvBlock(dec[2], dec[3])
        self.dfull = ConvBlock(dec[3] * 2, dec[3])
        self.head = nn.Conv2d(dec[3], out_ch, 1)

    def forward(self, x):
        stem = self.stem(x)
        f1, f2, f3, f4 = self.encoder(x)[-4:]
        d = self.d3(torch.cat([up_to(f4, f3), f3], 1))
        d = self.d2(torch.cat([up_to(d, f2), f2], 1))
        d = self.d1(torch.cat([up_to(d, f1), f1], 1))
        d = self.d0(F.interpolate(d, scale_factor=2, mode="bilinear", align_corners=False))
        d = self.dfull(torch.cat([up_to(d, stem), stem], 1))
        return self.head(d)


def loss_fn(logits, target, weight):
    bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    bce = (bce * weight).mean()
    prob = torch.sigmoid(logits)
    num = 2.0 * (prob * target).sum(dim=(2, 3)) + 1.0
    den = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + 1.0
    return bce + (1.0 - (num / den).mean())


_m = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=False).to(DEVICE).eval()
with torch.no_grad(), amp_autocast():
    _o = _m(torch.zeros(1, 3, PAD_H, PAD_W, device=DEVICE))
print("model forward ok:", tuple(_o.shape))
assert _o.shape[-2:] == (PAD_H, PAD_W)
del _m, _o; gc.collect(); torch.cuda.empty_cache()

model forward ok: (1, 2, 544, 704)


In [9]:
# ── 9 · Decode, official metric, fast evaluation ──────────────────────────────
def watershed_labels(fg, seed, t_cell, t_seed):
    mask = fg >= t_cell
    if not mask.any():
        return np.zeros(fg.shape, np.int32), 0
    markers, n = ndi.label((seed >= t_seed) & mask)
    if n == 0:
        markers, n = ndi.label(mask)
        if n == 0:
            return np.zeros(fg.shape, np.int32), 0
    lab = watershed(-fg, markers, mask=mask).astype(np.int32)
    return lab, int(lab.max())


def label_stats(lab, fg, seed, nlab):
    flat = lab.ravel()
    area = np.bincount(flat, minlength=nlab + 1).astype(np.float64)
    denom = np.maximum(area, 1.0)
    mean_fg = np.bincount(flat, weights=fg.ravel(), minlength=nlab + 1) / denom
    mean_seed = np.bincount(flat, weights=seed.ravel(), minlength=nlab + 1) / denom
    return area, mean_fg, mean_seed


def prepare_eval(fg, seed, gt_flats, t_cell, t_seed):
    lab, nlab = watershed_labels(fg, seed, t_cell, t_seed)
    area, mean_fg, mean_seed = label_stats(lab, fg, seed, nlab)
    n_gt = len(gt_flats)
    if nlab == 0 or n_gt == 0:
        return dict(n_gt=n_gt, area=area, score=mean_seed, gi=np.empty(0, int),
                    pi=np.empty(0, int), nlab=nlab)
    flatlab = lab.ravel()
    gt_area = np.array([f.size for f in gt_flats], np.float64)
    C = np.zeros((n_gt, nlab + 1), np.int64)
    for i, idx in enumerate(gt_flats):
        C[i] = np.bincount(flatlab[idx], minlength=nlab + 1)
    dice = 2.0 * C / np.maximum(gt_area[:, None] + area[None, :], 1e-9)
    dice[:, 0] = 0.0
    gi, pi = np.nonzero(dice >= 0.5)
    o = np.argsort(-dice[gi, pi])
    return dict(n_gt=n_gt, area=area, score=mean_seed, gi=gi[o], pi=pi[o], nlab=nlab)


def f1_filtered(prep, min_area, score_thr):
    keep = (prep["area"] >= min_area) & (prep["score"] >= score_thr)
    if keep.size:
        keep[0] = False
    n_pred = int(keep.sum())
    n_gt = prep["n_gt"]
    if n_gt == 0:
        return 1.0 if n_pred == 0 else 0.0
    if n_pred == 0:
        return 0.0
    used_g = np.zeros(n_gt, bool)
    used_p = np.zeros(keep.size, bool)
    tp = 0
    for g, p in zip(prep["gi"], prep["pi"]):
        if not keep[p] or used_g[g] or used_p[p]:
            continue
        used_g[g] = True; used_p[p] = True; tp += 1
    return 2.0 * tp / (2.0 * tp + (n_pred - tp) + (n_gt - tp))


def instances_from_maps(fg, seed, post):
    lab, nlab = watershed_labels(fg, seed, post["t_cell"], post["t_seed"])
    if nlab == 0:
        return []
    area, _, mean_seed = label_stats(lab, fg, seed, nlab)
    keep = (area >= post["min_area"]) & (mean_seed >= post.get("score_thr", 0.0))
    keep[0] = False
    flat = lab.ravel()
    order_ = np.argsort(flat, kind="stable")
    bounds = np.searchsorted(flat[order_], np.arange(nlab + 2))
    out = []
    for j in np.nonzero(keep)[0]:
        idx = order_[bounds[j]:bounds[j + 1]]
        if idx.size:
            out.append(np.sort(idx))
    return out


print("decode + metric ready")

decode + metric ready


In [10]:
# ── 10 · TTA inference and map I/O ────────────────────────────────────────────
@torch.no_grad()
def predict_maps(model, ids, image_dir, scales=None, flips=None):
    model.eval()
    scales = CFG.TTA_SCALES if scales is None else scales
    flips = CFG.TTA_FLIPS if flips is None else flips
    out = {}
    for image_id in ids:
        arr = IMAGES[image_id] if image_dir is None else load_gray_u8(image_dir / f"{image_id}.png")
        base = arr.astype(np.float32) / 255.0
        acc = np.zeros((2, CFG.IMG_H, CFG.IMG_W), np.float32)
        n = 0
        for s in scales:
            if abs(s - 1.0) < 1e-3:
                img = base
            elif HAVE_CV2:
                img = cv2.resize(base, (int(round(CFG.IMG_W * s)), int(round(CFG.IMG_H * s))),
                                 interpolation=cv2.INTER_LINEAR)
            else:
                img = np.asarray(Image.fromarray((base * 255).astype(np.uint8)).resize(
                    (int(round(CFG.IMG_W * s)), int(round(CFG.IMG_H * s))), Image.BILINEAR),
                    np.float32) / 255.0
            h, w = img.shape
            ph = int(math.ceil(h / CFG.PAD_MULT) * CFG.PAD_MULT)
            pw = int(math.ceil(w / CFG.PAD_MULT) * CFG.PAD_MULT)
            x = torch.from_numpy(np.ascontiguousarray(pad_to(img, ph, pw)))
            x = normalize_batch(x.unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)).to(DEVICE)
            variants = [(False, False)] if not flips else [(False, False), (True, False),
                                                           (False, True), (True, True)]
            for hf, vf in variants:
                xin = x
                if hf:
                    xin = torch.flip(xin, dims=[3])
                if vf:
                    xin = torch.flip(xin, dims=[2])
                with amp_autocast():
                    logits = model(xin)
                prob = torch.sigmoid(logits.float())
                if vf:
                    prob = torch.flip(prob, dims=[2])
                if hf:
                    prob = torch.flip(prob, dims=[3])
                p = prob[0, :, :h, :w].cpu().numpy()
                if p.shape[1:] != (CFG.IMG_H, CFG.IMG_W):
                    if HAVE_CV2:
                        p = np.stack([cv2.resize(p[c], (CFG.IMG_W, CFG.IMG_H),
                                                 interpolation=cv2.INTER_LINEAR) for c in range(2)])
                    else:
                        p = np.stack([np.asarray(Image.fromarray((p[c] * 255).astype(np.uint8)).resize(
                            (CFG.IMG_W, CFG.IMG_H), Image.BILINEAR), np.float32) / 255.0
                            for c in range(2)])
                acc += p
                n += 1
        out[image_id] = np.clip(acc / max(n, 1) * 255.0, 0, 255).astype(np.uint8)
    return out


def maps_path(member, split):
    if member in STAGE8_MEMBERS:
        return S8_DIR / f"{CFG.STAGE8_TAG}_{member}_{split}.npz"
    return MAPS / f"{CFG.RUN_TAG}_{member}_{split}.npz"


def load_maps(member, split):
    with np.load(maps_path(member, split)) as z:
        return {k: z[k] for k in z.files}


def family_mean(members, split, ids):
    acc = None
    for m in members:
        mp = load_maps(m, split)
        if acc is None:
            acc = {i: mp[i].astype(np.float32) for i in ids}
        else:
            for i in ids:
                acc[i] += mp[i].astype(np.float32)
        del mp; gc.collect()
    for i in ids:
        acc[i] /= (len(members) * 255.0)
    return acc


def maps_to_float(u8):
    return u8[0].astype(np.float32) / 255.0, u8[1].astype(np.float32) / 255.0


print("inference helpers ready")

inference helpers ready


In [11]:
# ── 11 · Train one probe ──────────────────────────────────────────────────────
def quick_val_f1(model, ids, post):
    maps = predict_maps(model, ids, None, scales=(1.0,), flips=False)
    return float(np.mean([
        f1_filtered(prepare_eval(*maps_to_float(maps[i]), INST[i], post["t_cell"], post["t_seed"]),
                    post["min_area"], post["score_thr"]) for i in ids]))


def train_probe(spec):
    tag = f"{CFG.RUN_TAG}_{spec['name']}"
    best_path = CKPT / f"{tag}_best.pt"
    if CFG.RESUME and best_path.exists():
        blob = torch.load(best_path, map_location="cpu")
        if blob.get("run_tag") == CFG.RUN_TAG and blob.get("member") == spec["name"]:
            print(f"[{spec['name']}] finished checkpoint (epoch {blob['epoch']}, "
                  f"F1 {blob['f1']:.4f}) - skipping")
            return blob["f1"]

    load_targets(spec["erosion"])
    seed_everything(CFG.PROBE_SEED)          # fixed across probes - erosion is the only variable
    model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=True).to(DEVICE)

    enc_params = list(model.encoder.parameters())
    enc_ids = {id(p) for p in enc_params}
    dec_params = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW(
        [{"params": dec_params, "lr": CFG.SUP_LR},
         {"params": enc_params, "lr": CFG.SUP_LR * CFG.SUP_ENCODER_LR_MULT}],
        weight_decay=CFG.SUP_WD)

    loader = DataLoader(CellDataset(train_ids, CFG.PROBE_CROP, True), batch_size=CFG.SUP_BATCH,
                        shuffle=True, num_workers=2, pin_memory=True, drop_last=True,
                        persistent_workers=True)
    total_steps = CFG.SUP_EPOCHS * CFG.STEPS_PER_EPOCH
    scaler = make_scaler()
    best_f1, best_epoch, log = -1.0, -1, []
    t0 = time.time()

    def lr_at(step):
        if step < CFG.WARMUP_STEPS:
            return (step + 1) / max(1, CFG.WARMUP_STEPS)
        p = (step - CFG.WARMUP_STEPS) / max(1, total_steps - CFG.WARMUP_STEPS)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))

    step, it = 0, iter(loader)
    for ep in range(CFG.SUP_EPOCHS):
        if time.time() - t0 > CFG.MEMBER_BUDGET_SEC:
            print(f"[{spec['name']}] budget reached at epoch {ep} - stopping")
            break
        freeze = ep < CFG.FREEZE_ENCODER_EPOCHS
        for p in enc_params:
            p.requires_grad_(not freeze)
        model.train()
        run = 0.0
        for _ in range(CFG.STEPS_PER_EPOCH):
            try:
                x, y, w, _ = next(it)
            except StopIteration:
                it = iter(loader)
                x, y, w, _ = next(it)
            x = normalize_batch(x).to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            w = w.to(DEVICE, non_blocking=True)
            mult = lr_at(step)
            opt.param_groups[0]["lr"] = CFG.SUP_LR * mult
            opt.param_groups[1]["lr"] = (CFG.SUP_LR * CFG.SUP_ENCODER_LR_MULT * mult
                                         * (0.0 if freeze else 1.0))
            opt.zero_grad(set_to_none=True)
            with amp_autocast():
                loss = loss_fn(model(x), y, w)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt); scaler.update()
            run += loss.item(); step += 1
        msg = f"[{spec['name']}] ep {ep + 1:02d}/{CFG.SUP_EPOCHS} loss {run / CFG.STEPS_PER_EPOCH:.4f}"
        if ep + 1 >= CFG.EVAL_START and (ep + 1 - CFG.EVAL_START) % CFG.EVAL_EVERY == 0:
            f1 = quick_val_f1(model, val_ids, CFG.DEFAULT_POST)
            msg += f" | val F1 {f1:.4f}"
            if f1 > best_f1:
                best_f1, best_epoch = f1, ep + 1
                torch.save({"run_tag": CFG.RUN_TAG, "member": spec["name"], "epoch": ep + 1,
                            "f1": f1, "state_dict": model.state_dict(), "spec": spec,
                            "encoder": CFG.ENCODER}, best_path)
                msg += "  *"
            log.append(dict(member=spec["name"], erosion=spec["erosion"], epoch=ep + 1,
                            loss=run / CFG.STEPS_PER_EPOCH, f1=f1))
        print(msg + f" | {fmt(time.time() - t0)}", flush=True)

    if best_f1 < 0:
        best_f1, best_epoch = quick_val_f1(model, val_ids, CFG.DEFAULT_POST), CFG.SUP_EPOCHS
        torch.save({"run_tag": CFG.RUN_TAG, "member": spec["name"], "epoch": best_epoch,
                    "f1": best_f1, "state_dict": model.state_dict(), "spec": spec,
                    "encoder": CFG.ENCODER}, best_path)
    pd.DataFrame(log).to_csv(ART / f"train_log_{spec['name']}.csv", index=False)
    print(f"[{spec['name']}] best epoch {best_epoch} | quick F1 {best_f1:.4f} | "
          f"{fmt(time.time() - t0)}")
    del model, opt, loader, scaler
    gc.collect(); torch.cuda.empty_cache()
    return best_f1

In [12]:
# ── 12 · Run the probes, cache maps ───────────────────────────────────────────
def write_submission(path, maps, ids, post):
    rows, counts, fallbacks = [], [], 0
    for image_id in ids:
        m = maps[image_id]
        fg, seed = maps_to_float(m) if m.dtype == np.uint8 else (m[0], m[1])
        inst = instances_from_maps(fg, seed, post)
        if not inst:  # an image with GT and zero masks scores exactly 0.0
            fallbacks += 1
            inst = instances_from_maps(fg, seed,
                                       dict(t_cell=0.30, t_seed=0.30, min_area=0, score_thr=0.0))[:1]
            if not inst:
                h = CFG.IMG_H // 2
                inst = [np.arange(h * CFG.IMG_W, h * CFG.IMG_W + 25)]
        counts.append(len(inst))
        rows.extend((image_id, flat_to_rle(f)) for f in inst)
    frame = pd.DataFrame(rows, columns=[ID_COL, ANN_COL])
    frame.insert(0, CFG.SUB_INDEX_COL, np.arange(len(frame), dtype=np.int64))
    frame.to_csv(path, index=False)
    assert set(frame[ID_COL].astype(str)) == set(ids), "Submission is missing test images."
    assert frame[ANN_COL].notna().all() and frame[CFG.SUB_INDEX_COL].is_unique
    return dict(path=str(path), rows=len(frame), mean=float(np.mean(counts)),
                min=int(np.min(counts)), max=int(np.max(counts)), fallbacks=fallbacks)


PROBE_RESULTS = []
for spec in CFG.MEMBERS:
    if elapsed() > CFG.TOTAL_BUDGET_SEC - CFG.FINAL_RESERVE_SEC:
        print(f"!! stopping after {len(PROBE_RESULTS)} probe(s) - reserving time for the sweep")
        break
    quick = train_probe(spec)
    tag = f"{CFG.RUN_TAG}_{spec['name']}"

    if CFG.RESUME and maps_path(spec["name"], "val").exists() \
            and maps_path(spec["name"], "test").exists():
        print(f"[{spec['name']}] cached maps found")
    else:
        blob = torch.load(CKPT / f"{tag}_best.pt", map_location="cpu")
        assert blob["run_tag"] == CFG.RUN_TAG and blob["member"] == spec["name"]
        model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS,
                     pretrained=False).to(DEVICE)
        model.load_state_dict(blob["state_dict"])
        t0 = time.time()
        np.savez_compressed(MAPS / f"{tag}_val.npz", **predict_maps(model, val_ids, None))
        np.savez_compressed(MAPS / f"{tag}_test.npz",
                            **predict_maps(model, TEST_IDS, TEST_DIR))
        print(f"[{spec['name']}] TTA maps cached in {fmt(time.time() - t0)}")
        del model, blob
        gc.collect(); torch.cuda.empty_cache()

    vm = load_maps(spec["name"], "val")
    tta = float(np.mean([
        f1_filtered(prepare_eval(*maps_to_float(vm[i]), INST[i], CFG.DEFAULT_POST["t_cell"],
                                 CFG.DEFAULT_POST["t_seed"]),
                    CFG.DEFAULT_POST["min_area"], CFG.DEFAULT_POST["score_thr"])
        for i in val_ids]))
    info = write_submission(ART / f"submission_{spec['name']}.csv",
                            load_maps(spec["name"], "test"), TEST_IDS, CFG.DEFAULT_POST)
    print(f"[{spec['name']}] erosion {spec['erosion']} | TTA F1 (default post) {tta:.4f} | "
          f"test preds/image {info['mean']:.1f}")
    PROBE_RESULTS.append(dict(member=spec["name"], erosion=spec["erosion"], quick_f1=quick,
                              tta_f1=tta, seed_pct=SEED_PCT.get(spec["erosion"]), **info))
    del vm; gc.collect()
    print("   >>> download /kaggle/working/ckpt/%s_best.pt now <<<\n" % tag, flush=True)

assert PROBE_RESULTS, "No probe finished."
POOL = [p["member"] for p in PROBE_RESULTS] + list(STAGE8_MEMBERS)
print("ensemble pool:", POOL, "| elapsed", fmt(elapsed()))

[e70] ep 01/70 loss 1.5363 | 0h 00m 34s
[e70] ep 02/70 loss 1.3978 | 0h 00m 43s
[e70] ep 03/70 loss 1.2989 | 0h 01m 12s
[e70] ep 04/70 loss 1.2350 | 0h 01m 25s
[e70] ep 05/70 loss 1.1735 | 0h 01m 38s
[e70] ep 06/70 loss 1.1233 | 0h 01m 52s
[e70] ep 07/70 loss 1.0721 | 0h 02m 05s
[e70] ep 08/70 loss 1.0276 | 0h 02m 19s
[e70] ep 09/70 loss 0.9857 | 0h 02m 33s
[e70] ep 10/70 loss 0.9583 | 0h 02m 47s
[e70] ep 11/70 loss 0.9145 | 0h 03m 01s
[e70] ep 12/70 loss 0.8790 | 0h 03m 14s
[e70] ep 13/70 loss 0.8474 | 0h 03m 28s
[e70] ep 14/70 loss 0.8118 | 0h 03m 42s
[e70] ep 15/70 loss 0.7800 | 0h 03m 55s
[e70] ep 16/70 loss 0.7511 | 0h 04m 09s
[e70] ep 17/70 loss 0.7198 | 0h 04m 23s
[e70] ep 18/70 loss 0.6876 | 0h 04m 36s
[e70] ep 19/70 loss 0.6510 | 0h 04m 50s
[e70] ep 20/70 loss 0.6260 | 0h 05m 04s
[e70] ep 21/70 loss 0.5967 | 0h 05m 18s
[e70] ep 22/70 loss 0.5774 | 0h 05m 31s
[e70] ep 23/70 loss 0.5527 | 0h 05m 45s
[e70] ep 24/70 loss 0.5332 | val F1 0.7720  * | 0h 06m 05s
[e70] ep 25/70 loss 0

In [13]:
# ── 13 · THE EROSION CURVE ────────────────────────────────────────────────────
# This is the deliverable. Everything else in the notebook is scaffolding for it.
def sweep(maps, ids, tc_grid, ts_grid, ma_grid, label=""):
    rows, per_image = [], {}
    t0 = time.time()
    for tc in tc_grid:
        for ts in ts_grid:
            preps = [prepare_eval(maps[i][0], maps[i][1], INST[i], tc, ts) for i in ids]
            for ma in ma_grid:
                for st in CFG.SCORE_THR_GRID:
                    vals = np.array([f1_filtered(p, ma, st) for p in preps])
                    rows.append(dict(t_cell=tc, t_seed=ts, min_area=ma, score_thr=st,
                                     f1=float(vals.mean()),
                                     mean_pred=float(np.mean([
                                         int(((p["area"][1:] >= ma) & (p["score"][1:] >= st)).sum())
                                         for p in preps]))))
                    per_image[(tc, ts, ma, st)] = vals
            del preps
    if label:
        print(f"    {label} swept in {time.time() - t0:.0f}s", flush=True)
    return pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True), per_image


def plateau_post(fine, tc_grid, ts_grid, ma_grid):
    # Never the peak: an optimum measured on 86 images is a region, not a point.
    peak = float(fine.f1.max())
    plat = fine[fine.f1 >= peak - CFG.ROBUST_TOL]
    snap = lambda v, grid: min(grid, key=lambda g: abs(g - v))
    return dict(t_cell=snap(float(np.median(plat.t_cell)), tc_grid),
                t_seed=snap(float(np.median(plat.t_seed)), ts_grid),
                min_area=int(snap(int(np.median(plat.min_area)), ma_grid)),
                score_thr=0.0)


coarse_ids = val_ids[:CFG.SWEEP_SUBSET]
KNOWN = {"n1": 0.45, "n2": 0.58}      # Stage 8, but at seed 2026/43 and different crops
curve = []
for m in POOL:
    vm = family_mean([m], "val", coarse_ids)
    tab, _ = sweep(vm, coarse_ids, CFG.COARSE_TC, CFG.COARSE_TS, CFG.COARSE_MA, label=m)
    b = tab.iloc[0]
    ero = next((p["erosion"] for p in PROBE_RESULTS if p["member"] == m), KNOWN.get(m))
    curve.append(dict(member=m, erosion=ero, coarse_f1=float(b.f1),
                      t_cell=float(b.t_cell), t_seed=float(b.t_seed),
                      min_area=int(b.min_area), mean_pred=float(b.mean_pred),
                      controlled=m in [p["member"] for p in PROBE_RESULTS]))
    print(f"  {m}: erosion {ero} | coarse F1 {b.f1:.4f} | preds/img {b.mean_pred:.1f} "
          f"| t_seed {b.t_seed:.2f}")
    del vm; gc.collect()

CURVE = pd.DataFrame(curve).sort_values("erosion").reset_index(drop=True)
CURVE.to_csv(ART / "erosion_curve.csv", index=False)
print("\n=== EROSION CURVE (coarse, %d images) ===" % len(coarse_ids))
print(CURVE.to_string(index=False))
print(f"\nGT mean instances/image: {STATS['gt_mean_instances']:.1f}")
print("Rows with controlled=True share seed 43 and crop 400x512, so their erosion")
print("comparison is clean. n1/n2 came from Stage 8 at different seeds/crops - they")
print("anchor the picture but are not part of the controlled curve.")

ctrl = CURVE[CURVE.controlled]
if len(ctrl) >= 2:
    top = ctrl.sort_values("coarse_f1", ascending=False).iloc[0]
    lo, hi = ctrl.erosion.min(), ctrl.erosion.max()
    if abs(top.erosion - hi) < 1e-9:
        print(f"\n!! best controlled probe is at the TOP of the tested range ({hi}) - "
              "the optimum may lie higher still; a further probe is warranted if time allows.")
    elif abs(top.erosion - lo) < 1e-9:
        print(f"\n!! best controlled probe is at the BOTTOM ({lo}) - "
              "higher erosion is not the answer; the knob is exhausted.")
    else:
        print(f"\n>>> interior optimum at erosion {top.erosion} - the curve is bracketed.")
    if ctrl.coarse_f1.max() - ctrl.coarse_f1.min() < 0.004:
        print("!! spread across the controlled probes is under 0.004 - that is inside noise. "
              "Treat the erosion knob as exhausted and do not spend more runs on it.")

    e70 swept in 23s
  e70: erosion 0.7 | coarse F1 0.8086 | preds/img 108.4 | t_seed 0.65
    e65 swept in 23s
  e65: erosion 0.65 | coarse F1 0.8049 | preds/img 108.3 | t_seed 0.55
    e80 swept in 22s
  e80: erosion 0.8 | coarse F1 0.8039 | preds/img 103.6 | t_seed 0.55
    e45 swept in 24s
  e45: erosion 0.45 | coarse F1 0.7933 | preds/img 111.1 | t_seed 0.70
    n1 swept in 22s
  n1: erosion 0.45 | coarse F1 0.7954 | preds/img 113.5 | t_seed 0.70
    n2 swept in 23s
  n2: erosion 0.58 | coarse F1 0.8035 | preds/img 108.4 | t_seed 0.60

=== EROSION CURVE (coarse, 86 images) ===
member  erosion  coarse_f1  t_cell  t_seed  min_area  mean_pred  controlled
   e45     0.45   0.793267    0.55    0.70        90 111.139535        True
    n1     0.45   0.795356    0.45    0.70        90 113.453488       False
    n2     0.58   0.803469    0.50    0.60        90 108.372093       False
   e65     0.65   0.804873    0.50    0.55        90 108.255814        True
   e70     0.70   0.808552    0

In [14]:
# ── 14 · Greedy forward selection, then a full sweep on the winner ────────────
# Combos are RANKED at one fixed reference setting (cheap), and only the winner gets
# the full grid. Stage 7 bought +0.0036 from two extra members and Stage 9 bought
# -0.0001 from five, so this stays opportunistic rather than becoming the plan.
REF = CURVE.sort_values("coarse_f1", ascending=False).iloc[0]
REF_POST = dict(t_cell=float(REF.t_cell), t_seed=float(REF.t_seed),
                min_area=int(REF.min_area), score_thr=0.0)
print("reference post for combo ranking:", REF_POST)


def combo_f1(members, ids, post):
    vm = family_mean(members, "val", ids)
    out = float(np.mean([
        f1_filtered(prepare_eval(vm[i][0], vm[i][1], INST[i], post["t_cell"], post["t_seed"]),
                    post["min_area"], post["score_thr"]) for i in ids]))
    del vm; gc.collect()
    return out


chosen = [str(REF.member)]
best_combo_f1 = combo_f1(chosen, val_ids, REF_POST)
print(f"start: {chosen} -> {best_combo_f1:.4f}")
improved = True
while improved and len(chosen) < len(POOL):
    improved = False
    cands = []
    for m in POOL:
        if m in chosen:
            continue
        f1 = combo_f1(chosen + [m], val_ids, REF_POST)
        cands.append((f1, m))
        print(f"  + {m}: {f1:.4f}")
    if cands:
        f1, m = max(cands)
        if f1 > best_combo_f1 + 1e-4:
            chosen.append(m)
            best_combo_f1 = f1
            improved = True
            print(f"  -> accepted {m} | {chosen} = {best_combo_f1:.4f}")
        else:
            print(f"  -> no member improves on {best_combo_f1:.4f}; stopping")

print(f"\nselected members: {chosen} | reference F1 {best_combo_f1:.4f}")
STATS["chosen_members"] = chosen

print("\nfull sweep on the selected combination ...", flush=True)
VAL_BEST = family_mean(chosen, "val", val_ids)
fine, PER = sweep(VAL_BEST, val_ids, CFG.T_CELL_GRID, CFG.T_SEED_GRID,
                  CFG.MIN_AREA_GRID, label="final")
fine.to_csv(ART / "sweep_final.csv", index=False)
POST = plateau_post(fine, CFG.T_CELL_GRID, CFG.T_SEED_GRID, CFG.MIN_AREA_GRID)
KEY = (POST["t_cell"], POST["t_seed"], POST["min_area"], POST["score_thr"])
if KEY not in PER:
    b = fine.iloc[0]
    POST = dict(t_cell=float(b.t_cell), t_seed=float(b.t_seed),
                min_area=int(b.min_area), score_thr=0.0)
    KEY = (POST["t_cell"], POST["t_seed"], POST["min_area"], POST["score_thr"])
LOCAL_F1 = float(PER[KEY].mean())
print(fine.head(10).to_string(index=False))
print(f"\nplateau centre {POST} | local F1 {LOCAL_F1:.4f} (peak {fine.f1.max():.4f})")
print("reference: Stage 8 ens 0.8073 -> 0.5294 public | Stage 9 blend 0.8072 -> 0.5286")

reference post for combo ranking: {'t_cell': 0.45, 't_seed': 0.65, 'min_area': 90, 'score_thr': 0.0}
start: ['e70'] -> 0.8086
  + e65: 0.8081
  + e80: 0.8023
  + e45: 0.8062
  + n1: 0.8074
  + n2: 0.8076
  -> no member improves on 0.8086; stopping

selected members: ['e70'] | reference F1 0.8086

full sweep on the selected combination ...
    final swept in 75s
 t_cell  t_seed  min_area  score_thr       f1  mean_pred
   0.45    0.65        90        0.0 0.808552 108.395349
   0.45    0.60        90        0.0 0.808400 108.837209
   0.40    0.60        90        0.0 0.807721 109.395349
   0.45    0.60        70        0.0 0.807672 111.232558
   0.40    0.65        90        0.0 0.807533 108.965116
   0.45    0.70        90        0.0 0.807262 107.720930
   0.50    0.65        90        0.0 0.807231 107.895349
   0.45    0.65       110        0.0 0.807217 105.662791
   0.40    0.65       110        0.0 0.807127 106.313953
   0.50    0.60        90        0.0 0.806999 108.279070

plateau 

In [15]:
# ── 15 · RLE parity, then write submission.csv ────────────────────────────────
parity = []
for image_id in val_ids[:25]:
    inst = instances_from_maps(VAL_BEST[image_id][0], VAL_BEST[image_id][1], POST)
    if not inst:
        parity.append(0.0); continue
    rt = [rle_decode_flat(flat_to_rle(f), CFG.IMG_H, CFG.IMG_W, RLE_ORDER) for f in inst]
    lab = np.zeros(NPIX, np.int32)
    for j, f in enumerate(rt, 1):
        lab[f] = j
    nlab = len(rt)
    gt = INST[image_id]
    gt_area = np.array([f.size for f in gt], np.float64)
    area = np.bincount(lab, minlength=nlab + 1).astype(np.float64)
    C = np.zeros((len(gt), nlab + 1), np.int64)
    for i, idx in enumerate(gt):
        C[i] = np.bincount(lab[idx], minlength=nlab + 1)
    dice = 2.0 * C / np.maximum(gt_area[:, None] + area[None, :], 1e-9)
    dice[:, 0] = 0.0
    gi, pi = np.nonzero(dice >= 0.5)
    o = np.argsort(-dice[gi, pi])
    ug, up, tp = set(), set(), 0
    for g_, p_ in zip(gi[o], pi[o]):
        if g_ in ug or p_ in up:
            continue
        ug.add(g_); up.add(p_); tp += 1
    parity.append(2.0 * tp / (2.0 * tp + (nlab - tp) + (len(gt) - tp)))

direct = float(PER[KEY][:25].mean())
delta = abs(direct - float(np.mean(parity)))
print(f"parity | in-memory {direct:.4f} | after RLE {np.mean(parity):.4f} | delta {delta:.5f}")
print("!! encoding is losing score - fix before submitting" if delta > 0.002
      else ">>> encoding is lossless")
STATS["parity_delta"] = float(delta)
del VAL_BEST; gc.collect()

TEST_BEST = family_mean(chosen, "test", TEST_IDS)
info = write_submission(WORK / "submission.csv", TEST_BEST, TEST_IDS, POST)
shutil.copy2(WORK / "submission.csv", ART / "submission_stage10.csv")
print("\nwrote:", WORK / "submission.csv", "|", info)
print(f"predictions/image {info['mean']:.1f} vs GT mean {STATS['gt_mean_instances']:.1f} "
      f"({100 * info['mean'] / STATS['gt_mean_instances']:.0f}% of the training rate)")
print("  Stage 8 sat at 111.3 (89%). If this has not moved, higher erosion did not")
print("  fix the under-counting and the remaining error is not a seed-size problem.")
STATS["submission"] = info
del TEST_BEST; gc.collect()

parity | in-memory 0.8251 | after RLE 0.8251 | delta 0.00000
>>> encoding is lossless

wrote: /kaggle/working/submission.csv | {'path': '/kaggle/working/submission.csv', 'rows': 19549, 'mean': 107.41208791208791, 'min': 13, 'max': 605, 'fallbacks': 0}
predictions/image 107.4 vs GT mean 125.2 (86% of the training rate)
  Stage 8 sat at 111.3 (89%). If this has not moved, higher erosion did not
  fix the under-counting and the remaining error is not a seed-size problem.


0

In [16]:
# ── 16 · Report ───────────────────────────────────────────────────────────────
probe_tab = pd.DataFrame(PROBE_RESULTS)[
    ["member", "erosion", "seed_pct", "quick_f1", "tta_f1", "mean", "fallbacks"]]
lines = [
    "# Stage 10 report - seed-erosion probe",
    f"- run tag: {CFG.RUN_TAG} | encoder: {CFG.ENCODER} (plain ImageNet-21k)",
    f"- all probes at seed {CFG.PROBE_SEED}, crop {CFG.PROBE_CROP} - erosion is the only variable",
    f"- fold {CFG.FOLD}/{CFG.N_FOLDS} | train/val {len(train_ids)}/{len(val_ids)} | "
    f"groups {STATS['n_groups']} | max val-to-train corr {STATS['max_val_train_corr']:.3f}",
    "",
    "## Probes",
] + ["    " + l for l in probe_tab.to_string(index=False).splitlines()] + [
    "",
    "## Erosion curve (coarse sweep)",
] + ["    " + l for l in CURVE.to_string(index=False).splitlines()] + [
    "",
    f"- selected members: {chosen}",
    f"- chosen post-processing: {POST}",
    f"- local F1: {LOCAL_F1:.4f}",
    f"- RLE parity delta: {STATS['parity_delta']:.5f}",
    f"- test predictions/image: mean {info['mean']:.1f} "
    f"(min {info['min']}, max {info['max']}), fallbacks {info['fallbacks']}",
    f"- GT mean instances/image: {STATS['gt_mean_instances']:.1f}",
    f"- total elapsed: {fmt(elapsed())}",
    "",
    "## Prior points",
    "| stage | local | public |",
    "|---|---|---|",
    "| 5  single ImageNet U-Net | 0.8279 | 0.5293 |",
    "| 7  SSL x3 ensemble       | 0.8035 | 0.5259 |",
    "| 8  non-SSL n1+n2         | 0.8073 | 0.5294 |",
    "| 9  cross-family blend    | 0.8072 | 0.5286 |",
    f"| 10 erosion probe        | {LOCAL_F1:.4f} | ? |",
    "",
    "Those four public scores span 0.0035 across four genuinely different models - that is the",
    "noise floor of a 91-image split, not a ranking. Judge this run on the erosion curve's",
    "shape, not on a single public score.",
    "",
    "## Selection",
    "Two slots. Keep the best measured file in one and something UNCORRELATED with it in the",
    "other - the slots are scored separately and the better private one counts, so two shots",
    "that fail differently beat two that fail together.",
]
(ART / "REPORT.md").write_text("\n".join(lines))
with open(ART / "stats.json", "w") as f:
    json.dump(STATS, f, indent=2, default=str)
print("\n".join(lines))
print("\nArtifacts:")
for p in sorted(ART.iterdir()):
    print(f"  {p.name:40s} {p.stat().st_size / 1e6:8.2f} MB")
print("\nDOWNLOAD /kaggle/working/ckpt/*.pt AND /kaggle/working/submission.csv NOW.")
print("Total elapsed:", fmt(elapsed()))

# Stage 10 report - seed-erosion probe
- run tag: stage10_erosion | encoder: convnext_tiny.fb_in22k_ft_in1k (plain ImageNet-21k)
- all probes at seed 43, crop (400, 512) - erosion is the only variable
- fold 0/5 | train/val 338/86 | groups 421 | max val-to-train corr 0.665

## Probes
    member  erosion  seed_pct  quick_f1   tta_f1       mean  fallbacks
       e70     0.70  1.561275  0.792441 0.806190 110.032967          0
       e65     0.65  1.968103  0.788588 0.804938 108.961538          0
       e80     0.80  0.778643  0.790789 0.791371 102.510989          0
       e45     0.45  4.169578  0.774600 0.790773 113.747253          0

## Erosion curve (coarse sweep)
    member  erosion  coarse_f1  t_cell  t_seed  min_area  mean_pred  controlled
       e45     0.45   0.793267    0.55    0.70        90 111.139535        True
        n1     0.45   0.795356    0.45    0.70        90 113.453488       False
        n2     0.58   0.803469    0.50    0.60        90 108.372093       False
       